# 🛠️ เทรนโมเดล YOLOv8 สำหรับระบบสแกนเครื่องมือ (Tool Scanner)
สมุดงาน Google Colab นี้ออกแบบมาสำหรับเทรนโมเดล **YOLOv8** เพื่อตรวจจับเครื่องมือช่างและตรวจสอบถาดอุปกรณ์

### 📌 ขั้นตอนทั้งหมด:
1. ติดตั้ง Dependencies (`ultralytics`)
2. ตรวจสอบ GPU (T4 / V100 / A100)
3. โหลดชุดข้อมูล (จาก ZIP ที่ Export มาจาก Admin หรือ Roboflow API)
4. เริ่มต้นเทรนโมเดล `yolov8m.pt`
5. ตรวจสอบความแม่นยำ (Validation mAP50)
6. ดาวน์โหลดไฟล์ `best.pt` กลับมาใส่ในระบบ Admin

## 1. ติดตั้ง Ultralytics และตรวจเช็ค GPU

In [ ]:
!pip install -q ultralytics roboflow

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU พร้อมใช้งาน: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ยังไม่ได้เปิด GPU! กรุณาไปที่ เมนู Runtime -> Change runtime type -> เลือก T4 GPU")

## 2. นำเข้า Dataset (เลือกวิธีใดวิธีหนึ่ง)

### วิธีที่ 2.1: ใช้ Roboflow API (แนะนำ - สะดวกสุด)

In [ ]:
# วางโค้ด Download จากหน้า Roboflow ของคุณที่นี่ (ตัวอย่าง):
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("tool-scanner")
# version = project.version(1)
# dataset = version.download("yolov8")

### วิธีที่ 2.2: อัปโหลดไฟล์ `yolo_dataset.zip` จากหน้า Admin โดยตรง

In [ ]:
# แตกไฟล์ ZIP ที่อัปโหลดขึ้น Colab
!unzip -q yolo_dataset.zip -d /content/dataset
!ls -la /content/dataset

## 3. เริ่มต้นเทรนโมเดล YOLOv8

In [ ]:
from ultralytics import YOLO

# เลือกรุ่น YOLOv8 Medium (สมดุลระหว่างความแม่นยำและความเร็ว)
model = YOLO("yolov8m.pt")

# เทรนโมเดล (ปรับ epochs ตามความเหมาะสม เช่น 50 - 100)
results = model.train(
    data="/content/dataset/data.yaml",  # ชี้ไปที่ไฟล์ data.yaml ของคุณ
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name="tool_scanner_v1"
)

## 4. ประเมินผลความแม่นยำ (Validation Metrics)

In [ ]:
# แสดงกราฟผลการเทรน (Loss, Precision, Recall, mAP50)
from IPython.display import Image
Image("runs/detect/tool_scanner_v1/results.png")

## 5. ดาวน์โหลดไฟล์น้ำหนักโมเดล `best.pt` กลับมาใช้งาน

In [ ]:
from google.colab import files

# ดาวน์โหลด best.pt สู่เครื่องของคุณเพื่อนำไปอัปโหลดในหน้า Admin
files.download("runs/detect/tool_scanner_v1/weights/best.pt")
print("✅ กำลังดาวน์โหลดไฟล์ best.pt เรียบร้อย!")